In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# --- 1. CONFIGURATION & FEATURES ---
# Use this switch to toggle between your Parquets and the DB later
USE_LIVE_DB = False 

# These are your Pertinent Columns (The signals)
FEATURES = [
    'net_interchange_mwh', 'balance_error', 'forecast_error_pct', 
    'demand_ramp_pct', 'demand_zscore', 'demand_residual',
    'temperature', 'precipitation', 'cloud_cover', 'wind_speed', 'shortwave_radiation',
    'weighted_temperature_2m', 'weighted_relative_humidity_2m', 'weighted_precipitation', 
    'weighted_cloud_cover', 'weighted_wind_speed_10m', 'weighted_shortwave_radiation',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_weekend',
    'population', 'latitude', 'longitude'
]

# --- 2. DATA ACCESS LAYER ---
def get_master_data(use_live_db=False):
    if use_live_db:
        # Once Postgres is ready, you'll put the connection and SQL query here
        # For now, this is a placeholder
        print("Connecting to PostgreSQL...")
        # conn = psycopg2.connect(...)
        # return pd.read_sql(your_query, conn)
        pass
    else:
        print("Merging local Parquet files into master matrix...")
        
        # Load Fact Tables
        energy_feat = pd.read_parquet('/workspaces/CECS-399-499/local_data/gold/fact_energy_features_hourly.parquet')
        energy_load = pd.read_parquet('/workspaces/CECS-399-499/local_data/gold/fact_energy_load_hourly.parquet')
        weather_city = pd.read_parquet('/workspaces/CECS-399-499/local_data/gold/fact_weather_city_hourly.parquet')
        
        # Load Dimension Tables
        time_dim = pd.read_parquet('/workspaces/CECS-399-499/local_data/gold/dim_time_hourly.parquet')
        city_dim = pd.read_parquet('/workspaces/CECS-399-499/local_data/gold/dim_city.parquet')
        
        # --- MERGE LOGIC ---
        # 1. Join Energy tables on time and source
        df = pd.merge(energy_feat, energy_load, on=['time_key', 'source_id'])
        
        # 2. Join Time details (brings in weekend/month info)
        df = pd.merge(df, time_dim, left_on='time_key', right_on='time_id')
        
        # 3. Join City/Weather details 
        # (Note: weather_city uses 'city_key', energy uses 'source_id')
        df = pd.merge(df, weather_city, on='time_key')
        df = pd.merge(df, city_dim, left_on='city_key', right_on='city_id')
        
        # 4. Handle Weighted Columns 
        # If your teammates haven't put these in a file yet, we can mock them 
        # or load them from a separate 'regional_weather.parquet'
        # df = pd.merge(df, regional_weather, on='time_key')

        return df

# --- 3. MODELING PIPELINE ---
# Load data using the switch
df_master = get_master_data(use_live_db=USE_LIVE_DB)

# Ensure all pertinent columns actually exist before training
# This prevents the model from crashing if a column was 'left out'
actual_features = [c for c in FEATURES if c in df_master.columns]

# Initialize and fit the Isolation Forest
# contamination=0.01 targets the top 1% most unusual hours
model = IsolationForest(contamination=0.005, random_state=42)

# Training (Only on the Pertinent Features)
df_master['anomaly_score'] = model.fit_predict(df_master[actual_features])

# --- 4. OUTPUT ---
# -1 = Anomaly, 1 = Normal
anomalies = df_master[df_master['anomaly_score'] == -1]
print(f"Detected {len(anomalies)} anomalies in the dataset.")

Merging local Parquet files into master matrix...
Detected 2192 anomalies in the dataset.


In [8]:
# Create a explicit copy so Pandas knows this is a standalone DataFrame
anomalies = anomalies.copy()

# Now adding the column won't trigger a warning
anomalies['date'] = pd.to_datetime(anomalies['time_key']).dt.date

# Alternatively, the .loc way (also stops the warning)
# anomalies.loc[:, 'date'] = pd.to_datetime(anomalies['time_key']).dt.date

# Load your known outages (Using the default engine now)
outages = pd.read_parquet('/workspaces/CECS-399-499/local_data/gold/fact_outage_daily.parquet')

# Update 'date_column' to the actual column name in your outage file (likely 'date')
outages['date'] = pd.to_datetime(outages['date']).dt.date 

# See how many anomalies happened on outage days
validated_anomalies = anomalies[anomalies['date'].isin(outages['date'])]

print(f"Validated {len(validated_anomalies)} anomalies against known outage days.")

Validated 75 anomalies against known outage days.


In [9]:
# Compare means
comparison = df_master.groupby('anomaly_score')[actual_features].mean()
print(comparison)

               net_interchange_mwh  balance_error  forecast_error_pct  \
anomaly_score                                                           
-1                   -43040.893704    -106.549726          117.404130   
 1                     -427.098704      -7.985178           -1.666059   

               demand_ramp_pct  demand_zscore  demand_residual  temperature  \
anomaly_score                                                                 
-1                  118.723992       0.747473     42288.421800    68.006661   
 1                   -0.831576       0.006998      -210.574792    60.494470   

               precipitation  cloud_cover  wind_speed  shortwave_radiation  \
anomaly_score                                                                
-1                  0.064280    45.536953    7.515785           483.253193   
 1                  0.006286    54.026972    6.099939           186.676336   

               hour_sin  hour_cos  month_sin  month_cos  is_weekend  \
anomal